# 🧠 Attention U-Net Road Extraction Model Training
### ISRO NNRMS Route Resilience Pipeline (Phase I Model)

This notebook trains the `AttentionUNet` model on Kaggle using a GPU. It is designed to:
1. **Auto-detect dataset locations** mounted in `/kaggle/input/` without manual configuration.
2. **Utilize Multi-GPU Parallelism**: Automatically detects and leverages both T4 GPUs on Kaggle via PyTorch's `nn.DataParallel` to accelerate training.
3. Use **Combined Loss** (40% Dice + 40% IoU + 10% BCE + 10% Boundary Loss) to output continuous, occlusion-robust road networks.
4. Apply **Albumentations augmentations** (shadows, fog, noise, coarse dropouts).
5. Save weights to `/kaggle/working/attention_unet_road.pth` for easy download.

> ℹ️ **Note on pip dependency conflicts**: When installing packages on Kaggle, you may see warnings about `cuml-cu12`, `cudf-cu12`, and `dask-cuda` conflicts. You can safely ignore these; they only affect RAPIDS AI libraries which are not used by this U-Net training pipeline.

In [ ]:
# ── Install/Verify Dependencies ──
# Note: The pip warnings about cuml, cudf, and dask-cuda can be safely ignored.
!pip install -q albumentations opencv-python-headless segmentation-models-pytorch

In [ ]:
import os
import glob
import math
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from PIL import Image
import matplotlib.pyplot as plt
import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm.notebook import tqdm

## 🔍 Step 1: Automatic Dataset Location Discovery
This cell searches `/kaggle/input` dynamically to find directories containing training images and their corresponding mask labels.

In [ ]:
def find_dataset_paths():
    input_dir = "/kaggle/input"
    print(f"Scanning {input_dir} for road dataset...")
    
    # Standard image patterns
    img_extensions = ["*.jpg", "*.jpeg", "*.png"]
    found_images = []
    for ext in img_extensions:
        found_images.extend(glob.glob(os.path.join(input_dir, "**", ext), recursive=True))
    
    if not found_images:
        # Fallback for local workspace tests
        print("⚠️ No images found in /kaggle/input. Setting up local mock directory for debugging.")
        os.makedirs("mock_data/images", exist_ok=True)
        os.makedirs("mock_data/masks", exist_ok=True)
        # Create a mock blank image
        Image.fromarray(np.zeros((512, 512, 3), dtype=np.uint8)).save("mock_data/images/mock_1.jpg")
        Image.fromarray(np.zeros((512, 512), dtype=np.uint8)).save("mock_data/masks/mock_1_mask.png")
        return os.path.abspath("mock_data/images"), os.path.abspath("mock_data/masks")
        
    # Check if there are separate folders containing 'mask' or 'label'
    folders = set(os.path.dirname(f) for f in found_images)
    mask_dirs = [d for d in folders if "mask" in d.lower() or "label" in d.lower()]
    image_dirs = [d for d in folders if d not in mask_dirs and ("image" in d.lower() or "sat" in d.lower() or "train" in d.lower())]
    
    if mask_dirs and image_dirs:
        return image_dirs[0], mask_dirs[0]
    
    # Check file-level naming convention
    mask_files = [f for f in found_images if "mask" in f.lower()]
    sat_files = [f for f in found_images if f not in mask_files]
    
    if mask_files and sat_files:
        return os.path.dirname(sat_files[0]), os.path.dirname(mask_files[0])

    # Absolute fallback to first folder containing images
    first_dir = os.path.dirname(found_images[0])
    return first_dir, first_dir

img_dir, mask_dir = find_dataset_paths()
print(f"\n🎉 Success! Detected Paths:")
print(f"📁 Satellite Images Directory: {img_dir}")
print(f"📁 Ground Truth Masks Directory: {mask_dir}")

## 🧠 Step 2: Attention U-Net Model Architecture
Includes channel attention, spatial attention, and skip-connection attention gates to reconstruct features under tree and shadow occlusions.

In [ ]:
class _DoubleConv(nn.Module):
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.net(x)

class _ChannelAttention(nn.Module):
    def __init__(self, channels: int, reduction: int = 16):
        super().__init__()
        mid = max(1, channels // reduction)
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Sequential(
            nn.Conv2d(channels, mid, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid, channels, 1, bias=False),
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg = self.fc(self.avg_pool(x))
        mx  = self.fc(self.max_pool(x))
        return x * self.sigmoid(avg + mx)

class _SpatialAttention(nn.Module):
    def __init__(self, kernel_size: int = 7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg = x.mean(dim=1, keepdim=True)
        mx, _ = x.max(dim=1, keepdim=True)
        attn = self.sigmoid(self.conv(torch.cat([avg, mx], dim=1)))
        return x * attn

class _AttentionGate(nn.Module):
    def __init__(self, g_ch: int, x_ch: int, mid_ch: int):
        super().__init__()
        self.Wg = nn.Conv2d(g_ch, mid_ch, 1, bias=True)
        self.Wx = nn.Conv2d(x_ch, mid_ch, 1, bias=True)
        self.psi = nn.Conv2d(mid_ch, 1, 1, bias=True)
        self.relu = nn.ReLU(inplace=True)
        self.sigmoid = nn.Sigmoid()

    def forward(self, g, x):
        g_up = F.interpolate(self.Wg(g), size=x.shape[2:], mode='bilinear', align_corners=False)
        x_t  = self.Wx(x)
        alpha = self.sigmoid(self.psi(self.relu(g_up + x_t)))
        return x * alpha

class AttentionUNet(nn.Module):
    def __init__(self, in_channels: int = 3, base_ch: int = 64):
        super().__init__()
        c = base_ch

        # Encoder
        self.enc1 = _DoubleConv(in_channels, c)
        self.enc2 = _DoubleConv(c,    c * 2)
        self.enc3 = _DoubleConv(c*2,  c * 4)
        self.enc4 = _DoubleConv(c*4,  c * 8)
        self.pool = nn.MaxPool2d(2)

        # Bottleneck
        self.bottleneck = _DoubleConv(c*8, c*16)
        self.ch_attn    = _ChannelAttention(c*16)
        self.sp_attn    = _SpatialAttention()
        
        # Decoder
        self.ag4 = _AttentionGate(c*16, c*8,  c*4)
        self.up4 = nn.ConvTranspose2d(c*16, c*8, 2, stride=2)
        self.dc4 = _DoubleConv(c*16, c*8)

        self.ag3 = _AttentionGate(c*8, c*4, c*2)
        self.up3 = nn.ConvTranspose2d(c*8, c*4, 2, stride=2)
        self.dc3 = _DoubleConv(c*8, c*4)

        self.ag2 = _AttentionGate(c*4, c*2, c)
        self.up2 = nn.ConvTranspose2d(c*4, c*2, 2, stride=2)
        self.dc2 = _DoubleConv(c*4, c*2)

        self.ag1 = _AttentionGate(c*2, c, c//2)
        self.up1 = nn.ConvTranspose2d(c*2, c, 2, stride=2)
        self.dc1 = _DoubleConv(c*2, c)

        self.head = nn.Conv2d(c, 1, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))

        b = self.bottleneck(self.pool(e4))
        b = self.ch_attn(b)
        b = self.sp_attn(b)

        d4 = self.dc4(torch.cat([self.ag4(b, e4),  self.up4(b)],  dim=1))
        d3 = self.dc3(torch.cat([self.ag3(d4, e3), self.up3(d4)], dim=1))
        d2 = self.dc2(torch.cat([self.ag2(d3, e2), self.up2(d3)], dim=1))
        d1 = self.dc1(torch.cat([self.ag1(d2, e1), self.up1(d2)], dim=1))

        return torch.sigmoid(self.head(d1))

## 📊 Step 3: Topologically-Aware Loss Functions
Combines Dice Loss, Jaccard/IoU Loss, standard BCE, and Laplacian Boundary Loss to capture fine, straight road segments.

In [ ]:
def dice_loss(pred: torch.Tensor, target: torch.Tensor, smooth: float = 1.0) -> torch.Tensor:
    pred   = pred.view(-1)
    target = target.view(-1)
    inter  = (pred * target).sum()
    return 1.0 - (2.0 * inter + smooth) / (pred.sum() + target.sum() + smooth)

def iou_loss(pred: torch.Tensor, target: torch.Tensor, smooth: float = 1.0) -> torch.Tensor:
    pred   = pred.view(-1)
    target = target.view(-1)
    inter  = (pred * target).sum()
    union  = pred.sum() + target.sum() - inter
    return 1.0 - (inter + smooth) / (union + smooth)

def boundary_loss(pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    lap_kernel = torch.tensor([[[[-1,-1,-1],[-1,8,-1],[-1,-1,-1]]]], dtype=torch.float32, device=pred.device)
    pred_b   = F.conv2d(pred,   lap_kernel, padding=1)
    target_b = F.conv2d(target.float(), lap_kernel, padding=1)
    return F.mse_loss(pred_b, target_b)

def combined_loss(pred: torch.Tensor, target: torch.Tensor,
                  w_dice: float = 0.4, w_iou: float = 0.4,
                  w_bce: float = 0.1, w_bnd: float = 0.1) -> torch.Tensor:
    bce = F.binary_cross_entropy(pred, target.float())
    bnd = boundary_loss(pred, target)
    dl  = dice_loss(pred, target)
    il  = iou_loss(pred, target)
    return w_dice * dl + w_iou * il + w_bce * bce + w_bnd * bnd

## 🩹 Step 4: Dataset & Augmentations
Creates a PyTorch Dataset loading images and masks, using Albumentations to simulate tree canopy occlusion, building shadows, fog, and sensor noise.

In [ ]:
class KaggleRoadDataset(Dataset):
    def __init__(self, img_dir, mask_dir, transform=None):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        
        all_files = sorted(os.listdir(img_dir))
        # Filters out masks if they reside in the same folder
        self.images = [f for f in all_files if f.lower().endswith(('.jpg', '.jpeg', '.png')) 
                       and 'mask' not in f.lower()]
        
        if not self.images:
            self.images = all_files
            
        self.transform = transform
        print(f"Initialized dataset with {len(self.images)} images.")

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]
        img_path = os.path.join(self.img_dir, img_name)
        
        # Auto-match matching masks
        base, ext = os.path.splitext(img_name)
        mask_candidates = [
            base + "_mask.png", base + "_mask.jpg", base + "-mask.png",
            base + ".png", img_name
        ]
        
        mask_path = None
        for cand in mask_candidates:
            p = os.path.join(self.mask_dir, cand)
            if os.path.exists(p):
                mask_path = p
                break
                
        if mask_path is None:
            for f in os.listdir(self.mask_dir):
                if f.startswith(base) and "mask" in f.lower():
                    mask_path = os.path.join(self.mask_dir, f)
                    break
                    
        if mask_path is None:
            mask_path = img_path
            
        image = np.array(Image.open(img_path).convert("RGB"))
        mask = np.array(Image.open(mask_path).convert("L"))
        mask = (mask > 127).astype(np.float32)
        
        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented["image"]
            mask = augmented["mask"].unsqueeze(0)
            
        return image, mask

def get_transforms(img_size: int = 512):
    train_transform = A.Compose([ 
        A.RandomResizedCrop(height=img_size, width=img_size, scale=(0.6, 1.0)),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.OneOf([
            A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=1.0),
            A.CLAHE(clip_limit=3.0, p=1.0),
        ], p=0.6),
        A.RandomShadow(p=0.3),
        # Canopy occlusion simulator
        A.CoarseDropout(max_holes=8, max_height=img_size//10, max_width=img_size//10, fill_value=0, p=0.4),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])
    
    val_transform = A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])
    return train_transform, val_transform

## 🚀 Step 5: Training Setup & Loop
Configures hyper-parameters, data splits, and trains the model with evaluation loops.

In [ ]:
def train_model(epochs=30, batch_size=8, lr=1e-4, img_size=512):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    
    train_t, val_t = get_transforms(img_size)
    full_dataset = KaggleRoadDataset(img_dir, mask_dir, transform=train_t)
    
    # Train / Val Split (85% / 15%)
    val_size = int(len(full_dataset) * 0.15)
    train_size = len(full_dataset) - val_size
    train_set, val_set = random_split(full_dataset, [train_size, val_size])
    
    # Re-apply non-augmented transforms to val set
    val_set.dataset.transform = val_t
    
    # Note: Use 4 workers for multi-GPU training throughput
    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
    val_loader   = DataLoader(val_set, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)
    
    model = AttentionUNet(in_channels=3, base_ch=64)
    if torch.cuda.device_count() > 1:
        print(f"🔥 Detected {torch.cuda.device_count()} GPUs. Using nn.DataParallel!")
        model = nn.DataParallel(model)
    model = model.to(device)
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    
    history = {"train_loss": [], "val_loss": [], "val_iou": []}
    best_val_loss = float("inf")
    
    for epoch in range(epochs):
        # Training phase
        model.train()
        train_loss = 0.0
        for images, masks in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]"):
            images = images.to(device)
            masks = masks.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = combined_loss(outputs, masks)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item() * images.size(0)
            
        train_loss /= len(train_set)
        scheduler.step()
        
        # Validation phase
        model.eval()
        val_loss = 0.0
        iou_score = 0.0
        with torch.no_grad():
            for images, masks in tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val]"):
                images = images.to(device)
                masks = masks.to(device)
                
                outputs = model(images)
                loss = combined_loss(outputs, masks)
                val_loss += loss.item() * images.size(0)
                
                # Metrics check
                preds = (outputs > 0.5).float()
                inter = (preds * masks).sum()
                union = preds.sum() + masks.sum() - inter
                iou_score += (inter / (union + 1e-6)).item() * images.size(0)
                
        val_loss /= len(val_set)
        iou_score /= len(val_set)
        
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_iou"].append(iou_score)
        
        print(f"Epoch {epoch+1:02d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val IoU: {iou_score:.4f}")
        
        # Save Best Checkpoint
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            state_dict = model.module.state_dict() if hasattr(model, 'module') else model.state_dict()
            torch.save(state_dict, "attention_unet_road.pth")
            print("⭐ Saved new best checkpoint to 'attention_unet_road.pth'")
            
    return history

## 📈 Step 6: Start Training & Plot Curves

In [ ]:
history = train_model(epochs=35, batch_size=8, lr=1.5e-4, img_size=512)

# Plot metrics
epochs_range = range(1, len(history["train_loss"]) + 1)
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, history["train_loss"], label="Training Loss")
plt.plot(epochs_range, history["val_loss"], label="Validation Loss")
plt.title("Loss Curves (Combined Loss)")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs_range, history["val_iou"], label="Validation IoU", color="green")
plt.title("Validation Intersection-over-Union (IoU)")
plt.xlabel("Epoch")
plt.ylabel("IoU")
plt.legend()

plt.tight_layout()
plt.savefig("training_curves.png")
plt.show()